In [ ]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler, TopicChunkPair, calculate_cohens_kappa
from reanimator.retrieval import Indexer, Retriever, reciprocal_rank_fusion, run_experiment
from reanimator.models import save_judgements, load_judgements

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
import pyterrier as pt
import nltk

load_dotenv()
nltk.download('punkt_tab')


reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

In [ ]:
#filter docs with original rel judgments to decrease candidate pool for faster processing in tutorial
#take only 40 docs and topic 42
only_k = 40
topic_id = "42"

human_judgements = reanimator.source.get_qrels()
topics = reanimator.source.get_topics()
topic = [t for t in topics if t.query_id == topic_id][0]

doc_ids = [judg.doc_id for judg in human_judgements if judg.query_id == topic_id]
len(doc_ids)

In [ ]:
docs = reanimator.load_documents(doc_ids=doc_ids)[:only_k]
reanimator.download_documents(docs)

#set accelerator options, device could be MPS, CUDA, CPU
accelerator_options = AcceleratorOptions(
        num_threads=8, device=AcceleratorDevice.MPS
    )

In [ ]:
docs = reanimator.load_documents(doc_ids=doc_ids)[:only_k]
reanimator.download_documents(docs)

In [ ]:
#only use docs where a pdf exists
docs = [doc for doc in docs if doc.pdf_path is not None]
print(f"Number of docs with pdf: {len(docs)}")

reanimator.extract_content(docs, accelerator_options)
reanimator.save_documents(docs, "data/documents")

In [ ]:
#generate chunks from downloaded documents 
#chunking config is at /src/reanimator/default_config.json
#config can be changed for exampke via reanimator.set_parameter("chunker.table_chunk_config.max_chars", 1000)

docs = reanimator.load_documents_from_file("data/documents")
chunks = reanimator.chunker.chunk(docs)

#chunks could also include metadata fields, e.g. abstract:
#chunks = reanimator.chunker.chunk(docs, metadata_fields_to_chunk=["abstract"])

In [ ]:
#set different chunk modality types

table_chunks = [c for c in chunks if c.modality == "table"]
text_chunks = [c for c in chunks if c.modality == "text"]

print(f"{len(table_chunks)} table chunks")
print(f"{len(text_chunks)} text chunks")

In [ ]:
#generate indices for different chunk modalities

indexer_both = Indexer(index_type="bm25", path="data/indices/bm25_both/")
indexer_table = Indexer(index_type="bm25", path="data/indices/bm25_table/")
indexer_text = Indexer(index_type="bm25", path="data/indices/bm25_text/")

indexer_both.index(chunks)
indexer_table.index(table_chunks)
indexer_text.index(text_chunks)

In [ ]:
retriever_both = Retriever(indexer=indexer_both)
retriever_table = Retriever(indexer=indexer_table)
retriever_text = Retriever(indexer=indexer_text)

In [ ]:
#generate pool of chunks from different retrieval systems for synthetic rel judgements

res_both = retriever_both.retrieve(topic=topic, k=20)
res_table = retriever_table.retrieve(topic=topic, k=20)
res_text = retriever_text.retrieve(topic=topic, k=20)

pool = set(reciprocal_rank_fusion([res_both, res_table, res_text]))
len(pool)

In [ ]:
#example closed source labeler
labeler_gpt41mini = OpenAILabeler(api_key=os.getenv("OPENAI_API_KEY"))

#example local model labeler (hosted via lm studio)
labeler_qwen = LocalModelLabeler(model="qwen/qwen3-30b-a3b", 
                                 base_url="http://localhost:1234/v1", 
                                 concurrency=10,
                                 thinking=False)



In [ ]:
#set batch to for labeling which appear in the pool

batch = [TopicChunkPair(topic=topic, chunk=chunk) for chunk in chunks if chunk.chunk_id in pool]
len(batch)

In [ ]:
#generate synthetic rel judgements

qwen_judgements = await labeler_qwen.label_all(batch)

model_name = labeler_qwen.model.replace("/", "_")
save_judgements(qwen_judgements, f"data/judgments/machine_{model_name}_judgements.json")

In [ ]:
gpt_judgements = await labeler_gpt41mini.label_all(batch)

model_name = labeler_gpt41mini.model.replace("/", "_")
save_judgements(gpt_judgements, f"data/judgments/machine_{model_name}_judgements.json")

In [ ]:
#calculate cohens kappa between two labelers

kappa = calculate_cohens_kappa("data/judgments/machine_gpt-4.1-mini-2025-04-14_judgements.json", "data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [ ]:
qwen_judgements = load_judgements("data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [ ]:
run_experiment(rankings=[res_both, res_table, res_text], topics=topics, judgements=qwen_judgements, eval_metrics=[pt.measures.nDCG, pt.measures.P@20], names=["BM25", "BM25 Table", "BM25 Text"])